In [18]:
import requests
import pandas as pd
import io

In [19]:
import os
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("API_KEY")
BASE_URL = os.getenv("BASE_URL")

In [20]:
headers = {
    "x-api-key": API_KEY,
    "Content-Type": "application/json"
}

# Diverse sample across industries
sample_domains = [
    # Tech / SaaS
    "stripe.com", "notion.so", "figma.com", "vercel.com", "atlassian.com",
    "github.com", "gitlab.com", "slack.com", "zoom.us", "dropbox.com",
    # Manufacturing
    "siemens.com", "bosch.com", "3m.com", "basf.com", "abb.com",
    # Healthcare / Pharma
    "novartis.com", "roche.com", "biontech.de", "bayer.com", "fresenius.com",
    # Finance / Fintech
    "n26.com", "ing.de", "commerzbank.de", "wirecard.com", "adyen.com",
    # Retail / E-commerce
    "zalando.de", "aboutyou.com", "otto.de", "mediamarkt.de", "rewe.de",
    # Consulting / Services
    "mckinsey.com", "bcg.com", "deloitte.com", "pwc.com", "accenture.com",
    # Logistics
    "dhl.com", "dpdhl.com", "rhenus.com", "hellmann.com", "db-schenker.com",
    # Energy
    "rwe.com", "eon.com", "vattenfall.de", "engie.com", "orsted.com",
    # Automotive
    "bmw.com", "mercedes-benz.com", "volkswagen.com", "porsche.com", "continental.com",
    # Construction
    "strabag.com", "skanska.com", "hochtief.com", "bilfinger.com", "goldbeck.com"
]

# Fetch in batches of 20
def fetch_batch(domains):
    payload = {
        "domains": domains,
        "columns": ["domain", "country", "state", "summary", "keywords"],
        "index": "webai*"
    }
    response = requests.post(BASE_URL, headers=headers, json=payload)
    return response.json()

batch_size = 20
all_results = []

for i in range(0, len(sample_domains), batch_size):
    batch = sample_domains[i:i+batch_size]
    print(f"Fetching batch {i//batch_size + 1}/{-(-len(sample_domains)//batch_size)}...")
    result = fetch_batch(batch)
    if "data" in result:
        all_results.extend(result["data"])

df = pd.DataFrame(all_results)
print(f"\nTotal fetched: {len(df)} companies")
print(df.head())

Fetching batch 1/3...
Fetching batch 2/3...
Fetching batch 3/3...

Total fetched: 43 companies
         domain      country        state  \
0  novartis.com  Switzerland  Basel-Stadt   
1        3m.com        China         None   
2     bayer.com        China         None   
3      basf.com        China         None   
4   siemens.com        China         None   

                                             summary  \
0  Novartis is an innovative medicines company fo...   
1  3M is a multinational conglomerate corporation...   
2  Bayer AG is a global enterprise with core comp...   
3  BASF is a global chemical company focused on c...   
4  Siemens Xcelerator Marketplace is a digital pl...   

                                            keywords  
0  , strategy, about novartis, vision, purpose  n...  
1                                               None  
2                                               None  
3                                               None  
4                     

In [21]:
# Analysis of the fetched data

# 1. Basic stats
print("=== BASIC STATS ===")
print(f"Total companies: {len(df)}")
print(f"Missing keywordss: {df['keywords'].isna().sum()} ({df['keywords'].isna().mean():.1%})")
print(f"Missing summarys: {df['summary'].isna().sum()} ({df['summary'].isna().mean():.1%})")
print(f"Missing state: {df['state'].isna().sum()} ({df['state'].isna().mean():.1%})")

# 2. keywords length distribution
df['desc_length'] = df['keywords'].fillna('').apply(len)
print("\n=== keywords LENGTH ===")
print(df['desc_length'].describe())
print(f"Empty keywordss (len=0): {(df['desc_length'] == 0).sum()}")

# 3. Country distribution
print("\n=== TOP COUNTRIES ===")
print(df['country'].value_counts().head(10))

# 4. Show problematic entries
print("\n=== MISSING keywordsS ===")
print(df[df['keywords'].isna()][['domain', 'country']])

# 5. Show shortest non-empty keywordss
print("\n=== SHORTEST keywordsS (potential quality issues) ===")
print(df[df['desc_length'] > 0].nsmallest(5, 'desc_length')[['domain', 'keywords']])

=== BASIC STATS ===
Total companies: 43
Missing keywordss: 30 (69.8%)
Missing summarys: 0 (0.0%)
Missing state: 19 (44.2%)

=== keywords LENGTH ===
count      43.000000
mean      158.139535
std       633.600429
min         0.000000
25%         0.000000
50%         0.000000
75%        61.500000
max      4084.000000
Name: desc_length, dtype: float64
Empty keywordss (len=0): 30

=== TOP COUNTRIES ===
country
China             18
Germany           10
United States      7
United Kingdom     3
Switzerland        1
Australia          1
Austria            1
Canada             1
Denmark            1
Name: count, dtype: int64

=== MISSING keywordsS ===
               domain         country
1              3m.com           China
2           bayer.com           China
3            basf.com           China
4         siemens.com           China
6       fresenius.com           China
7           roche.com           China
8             abb.com           China
9         biontech.de         Germany
10     

In [22]:
# Create a combined text field for embedding
df['combined_text'] = (
    df['summary'].fillna('') + ' ' + df['keywords'].fillna('')
).str.strip()

# How many have at least something usable?
df['has_text'] = df['combined_text'].str.len() > 20
print(f"Companies with usable text: {df['has_text'].sum()}/{len(df)}")

# Check combined length stats
df['combined_length'] = df['combined_text'].apply(len)
print("\n=== COMBINED TEXT LENGTH ===")
print(df['combined_length'].describe())

# Show the China/missing cases after combining
print("\n=== STILL EMPTY AFTER COMBINING ===")
print(df[~df['has_text']][['domain', 'country', 'summary', 'keywords']])

Companies with usable text: 43/43

=== COMBINED TEXT LENGTH ===
count      43.000000
mean      604.581395
std       636.623356
min       363.000000
25%       419.500000
50%       472.000000
75%       526.000000
max      4543.000000
Name: combined_length, dtype: float64

=== STILL EMPTY AFTER COMBINING ===
Empty DataFrame
Columns: [domain, country, summary, keywords]
Index: []


In [23]:
# Check the longest entry
longest = df.loc[df['combined_length'].idxmax()]
print("=== LONGEST ENTRY ===")
print(f"Domain: {longest['domain']}")
print(f"Length: {longest['combined_length']}")
print(f"Text preview:\n{longest['combined_text']}...")

# Check text quality visually for a few entries
print("\n=== SAMPLE TEXTS FOR EMBEDDING ===")
for _, row in df[df['has_text']].head(5).iterrows():
    print(f"\n[{row['domain']}]")
    print(row['combined_text'])
    print("-" * 50)

=== LONGEST ENTRY ===
Domain: strabag.com
Length: 4543
Text preview:
STRABAG SE is a leading European construction company that operates internationally across all sectors of the building industry. The company focuses on innovation, quality, and sustainability, offering comprehensive solutions from planning and construction to operation and demolition. STRABAG aims to shape living spaces and address future challenges with its 'People. Planet. Progress.' strategy, emphasizing human benefit and minimal environmental impact. STRABAG, STRABAG AG, Willkommen bei STRABAG, _STARTSEITE_NEUER SLIDER NEUER CLAIM, , Schienenverkehr, Abdichtung, Altlastensanierung, Asbestsanierung, Asphalt, Asphaltarbeiten, Asphaltproduktion, Asphaltwasserbau, Außenanlagen, Autobahn, Straßen, Autobahnbau, Bahnbau, Oberbau, Bahnbauwerke, Baustoffproduktion, Baustoffrecycling, Bauwerksabdichtung, Bauwerksbeläge, Bauwerkserhaltung, Bauwerksinstandsetzung, Belags- und Fräsarbeiten, Betondeckenfahrbahn, Betondeckenfert

In [24]:
import re

def clean_text(text):
    if not text:
        return ""
    
    # Remove navigation-style pipe-separated summarys
    # e.g. "ESG | Novartis | About | Novartis | News | Novartis"
    text = re.sub(r'(\b\w[\w\s]*\b\s*\|\s*){2,}', '', text)
    
    # Remove URLs
    text = re.sub(r'http\S+|www\.\S+', '', text)
    
    # Remove excessive whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Remove very short repeated words/fragments (navigation artifacts)
    text = re.sub(r'\b(\w+)\s+\1\b', r'\1', text)
    
    return text


In [25]:
# Apply cleaning
df['clean_text'] = df['combined_text'].apply(clean_text)
df['clean_length'] = df['clean_text'].apply(len)

# Compare before/after
print("=== BEFORE vs AFTER CLEANING ===")
print(f"Mean length before: {df['combined_length'].mean():.0f}")
print(f"Mean length after:  {df['clean_length'].mean():.0f}")

# Check stripe specifically
stripe = df[df['domain'] == 'stripe.com'].iloc[0]
print(f"\n=== STRIPE AFTER CLEANING ===")
print(stripe['clean_text'][:300])

# Check samples again
print("\n=== CLEANED SAMPLE TEXTS ===")
for _, row in df[df['has_text']].head(5).iterrows():
    print(f"\n[{row['domain']}]")
    print(row['clean_text'][:200])
    print("-" * 50)

=== BEFORE vs AFTER CLEANING ===
Mean length before: 605
Mean length after:  604

=== STRIPE AFTER CLEANING ===
Stripe is a financial infrastructure platform that enables businesses to accept payments, offer financial services, and implement custom revenue models. It provides a comprehensive suite of tools for online and in-person payments, billing, card issuing, and cross-border transactions. Stripe aims to 

=== CLEANED SAMPLE TEXTS ===

[novartis.com]
Novartis is an innovative medicines company focused on reimagining medicine to improve and extend people's lives. They develop and deliver transformative treatments for serious diseases, reaching mill
--------------------------------------------------

.com]
3M is a multinational conglomerate corporation that produces a wide range of products, including adhesives, abrasives, personal protective equipment, medical supplies, and electronic components. The c
--------------------------------------------------

[bayer.com]
Bayer AG is a glo

In [26]:
df.to_excel('istari_sample.xlsx', index=False)
print(f"Saved {len(df)} companies to istari_sample.xlsx")

Saved 43 companies to istari_sample.xlsx


In [27]:
import sys
print(sys.version)
print(sys.executable)

3.10.20 (main, Mar 11 2026, 17:43:48) [Clang 20.1.8 ]
/opt/anaconda3/envs/thesis/bin/python


In [28]:
# %pip install sentence-transformers
# %pip install faiss-cpu --no-cache-dir
# Alternative to faiss that's more lightweight
# %pip install sentence-transformers hnswlib
# !pip show sentence-transformers
# !pip show torch

In [29]:
from sentence_transformers import SentenceTransformer

In [30]:
model = SentenceTransformer('all-MiniLM-L6-v2')  # smaller model, less likely to crash

# Test on just 3 texts first
test_texts = ["This is a test company", "Another company description", "Third company"]
embeddings = model.encode(test_texts, show_progress_bar=True)

print(f"Success! Shape: {embeddings.shape}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10905.30it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 1/1 [00:00<00:00, 39.79it/s]

Success! Shape: (3, 384)


In [31]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# Load model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Encode all summaries
texts = df['summary'].tolist()
embeddings = model.encode(texts, show_progress_bar=True)
embeddings = np.array(embeddings).astype('float32')

print(f"Embeddings shape: {embeddings.shape}")
# Should be (43, 384)

# Build FAISS index
dimension = embeddings.shape[1]  # 384
index = faiss.IndexFlatL2(dimension)  # simple L2 distance index
index.add(embeddings)

print(f"Index built! Total vectors: {index.ntotal}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 14329.28it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 2/2 [00:00<00:00, 21.90it/s]

Embeddings shape: (43, 384)
Index built! Total vectors: 43


In [32]:
# Test query
query = "cloud software company for team collaboration"
query_embedding = model.encode([query]).astype('float32')

# Retrieve top 5
k = 5
distances, indices = index.search(query_embedding, k)

print(f"Top {k} results for: '{query}'\n")
for rank, (idx, dist) in enumerate(zip(indices[0], distances[0])):
    print(f"{rank+1}. {df.iloc[idx]['domain']} ({df.iloc[idx]['country']})")
    print(f"   Score: {dist:.4f}")
    print(f"   Summary: {df.iloc[idx]['summary'][:100]}...")
    print()

Top 5 results for: 'cloud software company for team collaboration'

1. github.com (United States)
   Score: 1.0476
   Summary: GitHub is a leading platform for software development, offering tools for collaboration, code manage...

2. gitlab.com (United States)
   Score: 1.1436
   Summary: GitLab is a comprehensive AI-powered DevSecOps platform that enables teams to build software faster ...

3. dropbox.com (United States)
   Score: 1.1678
   Summary: Dropbox is a cloud-based platform offering file hosting, synchronization, and collaboration services...

4. atlassian.com (United Kingdom)
   Score: 1.2117
   Summary: Atlassian is a software company that provides tools for software development, project management, an...

5. notion.so (United States)
   Score: 1.2338
   Summary: Notion is an AI-powered workspace designed to centralize knowledge, automate tasks, and manage proje...

